In [1]:
import dash
from dash import dcc, html, Input, Output, dash_table
import plotly.express as px
import pandas as pd
import networkx as nx


G = nx.Graph()
G.add_edges_from([
    ('A', 'B', {'cost': 10, 'time': 12}), ('A', 'C', {'cost': 5, 'time': 6}),
    ('C', 'B', {'cost': 5, 'time': 4}), ('A', 'D', {'cost': 2, 'time': 15}),
    ('D', 'B', {'cost': 11, 'time': 2}), ('C', 'E', {'cost': 8, 'time': 8}),
    ('E', 'B', {'cost': 4, 'time': 10}), ('A', 'E', {'cost': 15, 'time': 3}),
    ('D', 'C', {'cost': 3, 'time': 5})
])

def get_routes(start, end, mode):
    all_paths = list(nx.all_simple_paths(G, source=start, target=end))
    results = [{'Route': ' -> '.join(p), 'Cost': sum(G[p[i]][p[i+1]]['cost'] for i in range(len(p)-1)), 
                'Time': sum(G[p[i]][p[i+1]]['time'] for i in range(len(p)-1))} for p in all_paths]
    df = pd.DataFrame(results)
    if mode == 'Cost': return df.sort_values('Cost').head(1), df, "Strategy: Minimize Cost"
    if mode == 'Time': return df.sort_values('Time').head(1), df, "Strategy: Minimize Time"
    
  
    pareto = df.copy()
    for i, r1 in df.iterrows():
        for j, r2 in df.iterrows():
            if r2['Cost'] <= r1['Cost'] and r2['Time'] <= r1['Time'] and (r2['Cost'] < r1['Cost'] or r2['Time'] < r1['Time']):
                pareto = pareto.drop(i, errors='ignore'); break
    return pareto, df, "Strategy: Pareto-Optimal Trade-offs"


app = dash.Dash(__name__)

app.layout = html.Div(style={'padding': '20px', 'maxWidth': '1000px', 'margin': 'auto', 'fontFamily': 'system-ui'}, children=[
    html.Div(style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'space-between', 'marginBottom': '20px'}, children=[
        html.H3("CampusHelp Optimizer", style={'margin': '0'}),
        html.Div(id='driving-logic', style={'fontSize': '0.9em', 'color': '#666'})
    ]),
    
    html.Div(style={'display': 'grid', 'gridTemplateColumns': '250px 1fr', 'gap': '20px'}, children=[
        html.Div([
            html.Label("Start Building", style={'fontSize': '0.85em', 'fontWeight': '600'}),
            dcc.Dropdown(id='start', options=['A', 'B', 'C', 'D', 'E'], value='A', style={'marginBottom': '15px'}),
            html.Label("End Building", style={'fontSize': '0.85em', 'fontWeight': '600'}),
            dcc.Dropdown(id='end', options=['A', 'B', 'C', 'D', 'E'], value='B', style={'marginBottom': '15px'}),
            html.Label("Strategy", style={'fontSize': '0.85em', 'fontWeight': '600'}),
            dcc.Dropdown(id='mode', options=['Cost', 'Time', 'Pareto'], value='Pareto', clearable=False),
        ], style={'padding': '15px', 'background': '#f9f9f9', 'borderRadius': '8px', 'height': 'fit-content'}),
        
        html.Div([
            dcc.Graph(id='graph', config={'displayModeBar': False}, style={'height': '350px'}),
            dash_table.DataTable(id='table', page_size=5,
                                 style_cell={'textAlign': 'left', 'fontSize': '0.85em', 'padding': '8px'},
                                 style_header={'backgroundColor': '#eee', 'fontWeight': 'bold'})
        ])
    ])
])

@app.callback(
    [Output('graph', 'figure'), Output('table', 'data'), Output('driving-logic', 'children')],
    [Input('start', 'value'), Input('end', 'value'), Input('mode', 'value')]
)
def update(start, end, mode):
    opt_df, all_df, logic = get_routes(start, end, mode)
    

    fig = px.scatter(all_df, x='Cost', y='Time', color_discrete_sequence=['#bdc3c7'], 
                     hover_data=['Route'], opacity=0.6)
    fig.update_traces(marker=dict(size=12))
    

    fig.add_scatter(x=opt_df['Cost'], y=opt_df['Time'], mode='markers', 
                    marker=dict(size=18, color='#2c3e50' if mode != 'Pareto' else '#e74c3c'))
    
    fig.update_layout(plot_bgcolor='white', margin=dict(t=0, b=0, l=0, r=0), 
                      yaxis=dict(gridcolor='#eee', showline=True, linecolor='#eee'), 
                      xaxis=dict(gridcolor='#eee', showline=True, linecolor='#eee'))
    return fig, all_df.to_dict('records'), logic

if __name__ == '__main__':
    app.run(debug=True, dev_tools_ui=False)